# 🐼 Pandas Tricky Questions - Interactive Notebook
## Swiss Re Interview Prep

Run each cell and **predict the output** before executing!

In [1]:
import pandas as pd
import numpy as np
print(f"Pandas version: {pd.__version__}")

Pandas version: 2.2.2


---
## Q1: Chained Indexing (SettingWithCopyWarning)

In [4]:
# ❌ BAD: Chained indexing
df = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6]})

# This creates a VIEW, not a copy!
df[df["A"] > 1]["B"] = 100  # SettingWithCopyWarning!
print(df)  # B is NOT changed!

print("\nThe assignment didn't work!")

   A  B
0  1  4
1  2  5
2  3  6

The assignment didn't work!


/tmp/ipython-input-2722281421.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[df["A"] > 1]["B"] = 100  # SettingWithCopyWarning!


In [3]:
# ✅ GOOD: Use .loc
df = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6]})

df.loc[df["A"] > 1, "B"] = 100  # Correct!
print(df)

   A    B
0  1    4
1  2  100
2  3  100


---
## Q2: copy() vs Reference

In [5]:
# PREDICT: What happens to original?

original = pd.DataFrame({"A": [1, 2, 3]})
reference = original  # Just a reference!

reference["A"] = [10, 20, 30]

print("Original:")
print(original)

print("\nOriginal is ALSO modified!")

Original:
    A
0  10
1  20
2  30

Original is ALSO modified!


In [ ]:
# ✅ FIX: Use .copy()

original = pd.DataFrame({"A": [1, 2, 3]})
copy = original.copy()

copy["A"] = [10, 20, 30]

print("Original (unchanged):")
print(original)

print("\nCopy:")
print(copy)

---
## Q3: NaN Comparisons

In [ ]:
# PREDICT: What does this return?

df = pd.DataFrame({"A": [1, np.nan, 3]})

print(f"np.nan == np.nan: {np.nan == np.nan}")  # False!
print(f"\nFiltering with == np.nan:")
print(df[df["A"] == np.nan])  # Empty! NaN != NaN

In [ ]:
# ✅ Use isna() or isnull()

print("Using isna():")
print(df[df["A"].isna()])

print("\nUsing notna():")
print(df[df["A"].notna()])

In [ ]:
# More NaN gotchas:

s = pd.Series([1, np.nan, 3])
print(f"Sum: {s.sum()}")      # 4.0 (ignores NaN)
print(f"Mean: {s.mean()}")    # 2.0 (ignores NaN)
print(f"Count: {s.count()}")  # 2 (ignores NaN)
print(f"Size: {s.size}")      # 3 (includes NaN)

---
## Q4: Groupby Transform vs Apply

In [ ]:
df = pd.DataFrame({
    "group": ["A", "A", "B", "B"],
    "value": [1, 2, 3, 4]
})

# .agg() reduces to one row per group
print("agg (reduces):")
print(df.groupby("group").agg({"value": "sum"}))

# .transform() returns same shape as input
print("\ntransform (broadcasts back):")
df["group_sum"] = df.groupby("group")["value"].transform("sum")
print(df)

---
## Q5: merge (join) Gotchas

In [ ]:
# Watch for duplicates!

left = pd.DataFrame({"key": ["A", "A", "B"], "val1": [1, 2, 3]})
right = pd.DataFrame({"key": ["A", "A", "B"], "val2": [10, 20, 30]})

print("Left:")
print(left)
print("\nRight:")
print(right)

# How many rows after merge?
merged = pd.merge(left, right, on="key")
print(f"\nMerged ({len(merged)} rows):")
print(merged)

# 2 A's x 2 A's = 4 A rows + 1 B row = 5 rows!

In [ ]:
# Validate to catch duplicates:

try:
    merged = pd.merge(left, right, on="key", validate="one_to_one")
except pd.errors.MergeError as e:
    print(f"Caught error: {e}")

print("\nUse validate='one_to_one', 'one_to_many', 'many_to_one'")

---
## Q6: Boolean Indexing with Multiple Conditions

In [ ]:
df = pd.DataFrame({"A": [1, 2, 3, 4], "B": [5, 6, 7, 8]})

# ❌ WRONG: and/or don't work!
# df[(df["A"] > 1) and (df["B"] < 8)]  # ValueError!

# ✅ CORRECT: Use & | and parentheses!
print("AND condition:")
print(df[(df["A"] > 1) & (df["B"] < 8)])

print("\nOR condition:")
print(df[(df["A"] > 2) | (df["B"] < 6)])

print("\nNOT condition:")
print(df[~(df["A"] > 2)])

---
## Q7: inplace=True Gotcha

In [ ]:
# ❌ Common mistake:

df = pd.DataFrame({"A": [3, 1, 2]})

# This returns None!
sorted_df = df.sort_values("A", inplace=True)
print(f"sorted_df is: {sorted_df}")  # None!

print("\nOriginal is modified:")
print(df)

In [ ]:
# ✅ Better: Don't use inplace (deprecated pattern)

df = pd.DataFrame({"A": [3, 1, 2]})
sorted_df = df.sort_values("A")  # Returns new DataFrame
print(sorted_df)

---
## Q8: apply() Performance

In [ ]:
import time

df = pd.DataFrame({"A": range(100000)})

# ❌ SLOW: apply with Python function
start = time.time()
df["B"] = df["A"].apply(lambda x: x * 2)
print(f"apply: {time.time() - start:.4f}s")

# ✅ FAST: Vectorized operation
start = time.time()
df["C"] = df["A"] * 2
print(f"vectorized: {time.time() - start:.4f}s")

print("\nVectorized is 10-100x faster!")

---
## Q9: Index Reset

In [ ]:
df = pd.DataFrame({"A": [1, 2, 3, 4, 5]})

# Filter keeps original index
filtered = df[df["A"] > 2]
print("Filtered (old index):")
print(filtered)

# Reset to sequential
filtered = filtered.reset_index(drop=True)
print("\nReset index:")
print(filtered)

---
## Q10: Categorical Data

In [ ]:
# For columns with few unique values, use category!

# Regular string column
df = pd.DataFrame({"status": ["A", "B", "A", "C", "B", "A"] * 10000})
print(f"String memory: {df.memory_usage(deep=True).sum():,} bytes")

# Categorical
df["status"] = df["status"].astype("category")
print(f"Category memory: {df.memory_usage(deep=True).sum():,} bytes")

print("\nCategories reduce memory AND speed up groupby!")

---
## Summary Cheat Sheet

| Gotcha | Rule |
|--------|------|
| Chained indexing | Use `.loc[]` not `df[cond][col]` |
| Copy vs reference | Use `.copy()` when modifying |
| NaN comparison | Use `.isna()` not `== np.nan` |
| Transform vs Apply | Transform keeps shape, agg reduces |
| Merge duplicates | Use `validate=` parameter |
| Boolean operators | Use `& | ~` not `and or not` |
| inplace=True | Returns None, avoid using |
| apply() | Avoid, use vectorized ops |